In [ ]:
import mysql.connector
import numpy as np
import pandas as pd
import astropy.io.fits as fits
from matplotlib import pyplot as plt
import tiledb
import os

In [2]:
conn = mysql.connector.connect(
    user='root',
    password='',
    unix_socket='/tmp/mysql_database_dev.sock',
)


In [3]:
cur = conn.cursor()
cur.execute("CREATE DATABASE new_heat_db")

In [4]:
cur.execute("show databases")
for db in cur:
    print(db)


('HEAT_DB',)
('information_schema',)
('mysql',)
('new_heat_db',)
('performance_schema',)
('sys',)
('TIRI_pre',)


In [ ]:
cur.execute("use new_heat_db")

In [9]:
img_file = "./hyb2_tir_20180801_101707_l1.fit"
hdul = fits.open(img_file)
hdul.info()

Filename: ./hyb2_tir_20180801_101707_l1.fit
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU      38   (384, 256)   float32   


In [18]:
hdu = hdul[0]
header = hdu.header
schema_df = pd.DataFrame(columns=["key", "value_type"])
for key, value in header.items():
    print(f"{key}: {type(value)}")
    schema_df = pd.concat([schema_df, pd.DataFrame({"key": [key], "value_type": [type(value)]})], ignore_index=True)
schema_df.to_csv("fits_header.csv", index=False)
    
    


SIMPLE: <class 'bool'>
BITPIX: <class 'int'>
NAXIS: <class 'int'>
NAXIS1: <class 'int'>
NAXIS2: <class 'int'>
EXTEND: <class 'bool'>
COMMENT: <class 'str'>
COMMENT: <class 'str'>
ORIGIN: <class 'str'>
DATE-BEG: <class 'str'>
DATE-OBS: <class 'str'>
DATE-END: <class 'str'>
TELESCOP: <class 'str'>
SPCECRFT: <class 'str'>
INSTRUME: <class 'str'>
OBJECT: <class 'str'>
BUNIT: <class 'str'>
ROI_LLX: <class 'int'>
ROI_LLY: <class 'int'>
ROI_URX: <class 'int'>
ROI_URY: <class 'int'>
BITDEPTH: <class 'int'>
BOL_TEMP: <class 'float'>
PKG_TEMP: <class 'float'>
CAS_TEMP: <class 'float'>
SHT_TEMP: <class 'float'>
LEN_TEMP: <class 'float'>
PLT_RDYC: <class 'str'>
PLT_RDYF: <class 'str'>
PLT_TGTT: <class 'str'>
PLT_POW: <class 'str'>
IMGTYPE: <class 'str'>
IMGCMPRV: <class 'str'>
IMGCMPAL: <class 'str'>
IMGCMPPR: <class 'str'>
IMGCRRPT: <class 'str'>
IMGACCM: <class 'int'>
VERSION: <class 'float'>


In [ ]:
# テーブル定義と一致させるため、重複キーは COMMENT_1 のように連番を付ける
key_count = {}
df_header = pd.DataFrame(index=[0])
for key, value in header.items():
    if key not in key_count:
        key_count[key] = 0
        col = key
    else:
        key_count[key] += 1
        col = f"{key}_{key_count[key]}"
    df_header[col] = value
filename = img_file.replace("./", "")
df_header.insert(loc=0, column="filename", value=filename)
print(df_header)

                          filename  SIMPLE  BITPIX  NAXIS  NAXIS1  NAXIS2  \
0  hyb2_tir_20180801_101707_l1.fit    True     -32      2     384     256   

   EXTEND                                            COMMENT     ORIGIN  \
0    True    and Astrophysics', volume 376, page 359; bib...  ISAS/JAXA   

                  DATE-BEG  ... PLT_RDYF PLT_TGTT PLT_POW IMGTYPE  IMGCMPRV  \
0  2018-08-01T10:17:06.248  ...     FINE  40 degC      ON     PIC  LOSSLESS   

     IMGCMPAL IMGCMPPR  IMGCRRPT  IMGACCM  VERSION  
0  STAR_PIXEL     0x30        OK       32      0.3  

[1 rows x 38 columns]


In [30]:
def python_type_to_mysql(type_str):
    if "bool" in type_str:
        return "TINYINT(1)"
    elif "int" in type_str:
        return "INT"
    elif "float" in type_str:
        return "DOUBLE"
    elif "str" in type_str:
        return "VARCHAR(512)"
    else:
        return "VARCHAR(512)"

In [34]:
# 重複カラム名（COMMENT など）をユニークにする
used_names = {"filename"}
columns = ["filename VARCHAR(255) PRIMARY KEY"]
for _, row in schema_df.iterrows():
    key = row["key"]
    col_name = key
    if col_name in used_names:
        i = 1
        while f"{key}_{i}" in used_names:
            i += 1
        col_name = f"{key}_{i}"
    used_names.add(col_name)
    mysql_type = python_type_to_mysql(str(row["value_type"]))
    columns.append(f"`{col_name}` {mysql_type}")

In [35]:
create_sql = f"CREATE TABLE fits_header (\n  " + ",\n  ".join(columns) + "\n)"
print(create_sql)

CREATE TABLE fits_header (
  filename VARCHAR(255) PRIMARY KEY,
  `SIMPLE` TINYINT(1),
  `BITPIX` INT,
  `NAXIS` INT,
  `NAXIS1` INT,
  `NAXIS2` INT,
  `EXTEND` TINYINT(1),
  `COMMENT` VARCHAR(512),
  `COMMENT_1` VARCHAR(512),
  `ORIGIN` VARCHAR(512),
  `DATE-BEG` VARCHAR(512),
  `DATE-OBS` VARCHAR(512),
  `DATE-END` VARCHAR(512),
  `TELESCOP` VARCHAR(512),
  `SPCECRFT` VARCHAR(512),
  `INSTRUME` VARCHAR(512),
  `OBJECT` VARCHAR(512),
  `BUNIT` VARCHAR(512),
  `ROI_LLX` INT,
  `ROI_LLY` INT,
  `ROI_URX` INT,
  `ROI_URY` INT,
  `BITDEPTH` INT,
  `BOL_TEMP` DOUBLE,
  `PKG_TEMP` DOUBLE,
  `CAS_TEMP` DOUBLE,
  `SHT_TEMP` DOUBLE,
  `LEN_TEMP` DOUBLE,
  `PLT_RDYC` VARCHAR(512),
  `PLT_RDYF` VARCHAR(512),
  `PLT_TGTT` VARCHAR(512),
  `PLT_POW` VARCHAR(512),
  `IMGTYPE` VARCHAR(512),
  `IMGCMPRV` VARCHAR(512),
  `IMGCMPAL` VARCHAR(512),
  `IMGCMPPR` VARCHAR(512),
  `IMGCRRPT` VARCHAR(512),
  `IMGACCM` INT,
  `VERSION` DOUBLE
)


In [36]:
cur.execute(create_sql)
cur.execute("DROP TABLE IF EXISTS fits_header")
cur.execute(create_sql)
print("\nテーブル fits_header を作成しました")


テーブル fits_header を作成しました


In [41]:
# df_header は fits_header テーブルと同形式（ワイド形式）
# カラム名をバッククォートで囲む（予約語・ハイフン対策）
cols = ",".join([f"`{c}`" for c in df_header.columns])
placeholders = ",".join(["%s"] * len(df_header.columns))
sql = f"INSERT INTO fits_header ({cols}) VALUES ({placeholders})"

for row in df_header.itertuples(index=False):
    cur.execute(sql, tuple(row))

conn.commit()
print("挿入完了")

ProgrammingError: 1064 (42000): You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 'key,value) VALUES ('hyb2_tir_20180801_101707_l1.fit','SIMPLE',True)' at line 1